# Uber Driver Signup Conversion — GAMMA Analysis

**Goal:** Predict which driver signups will complete their first trip (binary classification).  
Explore conversion patterns and suggest operational strategies.

**Business Questions:**
1. What fraction of signups completed a first trip?
2. Build a predictive model; discuss approach
3. How might Uber operationalise insights?

## 0. 環境設定

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from gamma import GAMMA_DNA
from gamma.data_exploration import gamma_de_load_files

print('Environment ready.')

Environment ready.


## 1. 資料載入與 ETL

In [2]:
# Load raw data
df = gamma_de_load_files('datasets/ds_challenge_v2_1_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (54681, 11)


,id,city_name,signup_os,signup_channel,signup_date,bgc_date,vehicle_added_date,vehicle_make,vehicle_model,vehicle_year,first_completed_date
0,1,Strark,ios web,Paid,1/2/16,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Strark,windows,Paid,1/21/16,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Wrouver,windows,Organic,1/11/16,1/11/16,NaN,NaN,NaN,NaN,NaN
3,4,Berton,android web,Referral,1/29/16,2/3/16,2/3/16,Toyota,Corolla,2016.0,2/3/16
4,5,Strark,android web,Referral,1/10/16,1/25/16,1/26/16,Hyundai,Sonata,2016.0,NaN


In [3]:
# Create binary target: 1 = completed first trip, 0 = did not
df['first_trip'] = (df['first_completed_date'].notna()).astype(int)

conversion_rate = df['first_trip'].mean()
print(f'Overall conversion rate: {conversion_rate:.1%}')
print(df['first_trip'].value_counts())

Overall conversion rate: 11.2%
first_trip
0    48544
1     6137
Name: count, dtype: int64


In [4]:
# Initialise GAMMA
g = GAMMA_DNA(
    df,
    target='first_trip',
    task='binary_classification',
    target_label=('converted', 'not_converted'),
    name='uber_drivers'
)
g.skim()

,variable,dtype,n,n_missing,pct_missing,n_unique,mean,std,se,min,p25,median,p75,max,skew,kurtosis,top,top_freq
0,id,int64,54681,0,0.00,54681,27341.0000,15785.1894,67.5043,1.0,13671.0,27341.0,41011.0,54681.0,0.0000,-1.2000,NaN,NaN
1,city_name,object,54681,0,0.00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Strark,29557.0
2,signup_os,object,47824,6857,12.54,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ios web,16632.0
3,signup_channel,object,54681,0,0.00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Paid,23938.0
4,signup_date,object,54681,0,0.00,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1/5/16,2489.0
5,bgc_date,object,32896,21785,39.84,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1/29/16,1125.0
6,vehicle_added_date,object,13134,41547,75.98,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1/26/16,377.0
7,vehicle_make,object,13223,41458,75.82,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Toyota,3219.0
8,vehicle_model,object,13223,41458,75.82,368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Civic,689.0
9,vehicle_year,float64,13223,41458,75.82,24,2010.5680,35.2192,0.3063,0.0,2008.0,2013.0,2015.0,2017.0,-56.2863,3210.3178,NaN,NaN


## 2. Pipeline + Feature Engineering

In [5]:
def feature_fn(df):
    df = df.copy()

    # Parse dates — signup_date uses 'YYYY MM DD' format (spaces not dashes)
    df['signup_date_parsed'] = pd.to_datetime(
        df['signup_date'].str.replace(' ', '-'), errors='coerce'
    )
    df['bgc_date_parsed'] = pd.to_datetime(df['bgc_date'], errors='coerce')
    df['vehicle_added_date_parsed'] = pd.to_datetime(df['vehicle_added_date'], errors='coerce')
    df['first_completed_date_parsed'] = pd.to_datetime(df['first_completed_date'], errors='coerce')

    # Days from signup to background check
    df['days_to_bgc'] = (
        df['bgc_date_parsed'] - df['signup_date_parsed']
    ).dt.days

    # Days from signup to vehicle added
    df['days_to_vehicle'] = (
        df['vehicle_added_date_parsed'] - df['signup_date_parsed']
    ).dt.days

    # Signup month and day of week
    df['signup_month'] = df['signup_date_parsed'].dt.month
    df['signup_dayofweek'] = df['signup_date_parsed'].dt.dayofweek

    # Vehicle age (dataset circa 2016)
    df['vehicle_age'] = 2016 - pd.to_numeric(df['vehicle_year'], errors='coerce')

    # Completion indicators
    df['bgc_completed'] = df['bgc_date_parsed'].notna().astype(int)
    df['vehicle_added'] = df['vehicle_added_date_parsed'].notna().astype(int)

    return df

g.pipe('featured', feature_fn).run(from_stage='raw')
g.warehouse.persist('.warehouse/uber')

WindowsPath('.warehouse/uber')

## 3. EDA — 詳盡分析

### Q1: Overall Conversion Rate

In [6]:
featured_df = g.frames['featured'].copy()

conversion_rate = featured_df['first_trip'].mean()
print(f'Overall driver conversion rate: {conversion_rate:.1%}')
print(f'  Converted:     {featured_df["first_trip"].sum():,}')
print(f'  Not converted: {(featured_df["first_trip"] == 0).sum():,}')

Overall driver conversion rate: 11.2%
  Converted:     6,137
  Not converted: 48,544


In [7]:
# Auto-explore categorical features
g.viz.auto_explore(['signup_os', 'signup_channel', 'city_name'])

In [8]:
# Conversion rate by signup OS
g.rate_by('signup_os').plot()

In [9]:
# Conversion rate by signup channel
g.rate_by('signup_channel').plot()

In [10]:
# Conversion rate by city
g.rate_by('city_name').plot()

In [11]:
# Distribution of days to background check
g.viz.hist('days_to_bgc')

【days_to_bgc】統計量（已去除NA，clip於[0.01, 0.99]分位數之間）:


,count,mean,std,min,1%,25%,50%,75%,99%,max
days_to_bgc,"32,896",10,11,0,0,2,6,15,43,69


In [12]:
# Distribution of days to vehicle added
g.viz.hist('days_to_vehicle')

【days_to_vehicle】統計量（已去除NA，clip於[0.01, 0.99]分位數之間）:


,count,mean,std,min,1%,25%,50%,75%,99%,max
days_to_vehicle,"13,134",15,14,-5,0,4,11,24,53,72


In [13]:
# Scatter: days_to_bgc vs days_to_vehicle
g.viz.scatter('days_to_bgc', 'days_to_vehicle')

In [14]:
# Scatter: vehicle_age vs first_trip
g.viz.scatter('vehicle_age', 'first_trip')

In [15]:
# EDA with segmentation
eda = g.eda(segment_cols=['signup_channel', 'signup_os'])
eda.top_signals()

🔍 Running EDA pipeline...
  [1/5] Inspecting structure...


  [2/5] Inspecting quality...


  [3/5] Analysing missingness...


  [4/5] Analysing feature → 'first_trip' relationships...


  [5/5] Analysing redundancy...
✅ EDA complete.



,feature,signal_score,relationship_strength,monotonicity_score,lift_max,interpretation
0,vehicle_added,60.87,0.5964,1.000,1.000,Feature `vehicle_added` shows a strong positiv...
1,days_to_vehicle,58.37,0.5866,1.000,1.815,Feature `days_to_vehicle` shows a strong negat...
2,days_to_bgc,54.14,0.3057,1.000,2.113,Feature `days_to_bgc` shows a strong negative ...
3,bgc_completed,50.13,0.2893,1.000,1.000,Feature `bgc_completed` shows a moderate posit...
4,signup_dayofweek,32.05,0.0364,0.800,1.117,Feature `signup_dayofweek` shows a weak negati...
5,vehicle_age,19.55,0.0224,0.714,1.097,Feature `vehicle_age` shows a weak negative br...
6,vehicle_year,19.50,0.0224,0.714,1.089,Feature `vehicle_year` shows a weak positive b...
7,id,19.48,0.0228,0.556,1.132,Feature `id` shows a weak negative non-monoton...
8,city_name,NaN,0.0046,NaN,1.079,Feature `city_name` shows a weak association ...
9,signup_os,NaN,0.0114,NaN,1.450,Feature `signup_os` shows a weak association ...


In [16]:
# Correlation heatmap
g.correlation_heatmap()

vehicle_added       0.596446
bgc_completed       0.289346
vehicle_year        0.022391
id                 -0.009360
vehicle_age        -0.022391
signup_dayofweek   -0.036409
days_to_bgc        -0.305676
days_to_vehicle    -0.586626
signup_month             NaN
Name: first_trip, dtype: float64

In [17]:
# Leakage check — first_completed_date should be flagged
leakage = g.leakage()
leakage.summary()


  Leakage Detection Report (target='first_trip')
  Features checked    : 22
  High-severity flags : 7
  Medium flags        : 0

  🔴 [HIGH] city_name
     Reason         : Feature name matches common target-leakage patterns (score, proba, flag, etc.).
     Evidence       : Name pattern match: 'city_name'
     Recommendation : Review whether this column is derived from or correlated with the target. Drop if it encodes post-event information.

  🔴 [HIGH] vehicle_year
     Reason         : Extremely high Information Value — this level of predictive power is unusual and may indicate target leakage.
     Evidence       : IV = 3.8960 ≥ threshold 0.5
     Recommendation : Review data lineage for this column. IV > 0.5 is very strong; IV > 1.0 almost always indicates leakage.

  🔴 [HIGH] days_to_bgc
     Reason         : Extremely high Information Value — this level of predictive power is unusual and may indicate target leakage.
     Evidence       : IV = 11.1318 ≥ threshold 0.5
     Recommend

In [18]:
# Pandas groupby: conversion rate by city, channel, os
for col in ['city_name', 'signup_channel', 'signup_os']:
    print(f'\n=== Conversion rate by {col} ===')
    summary = (
        featured_df.groupby(col)['first_trip']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'conversion_rate', 'count': 'n'})
        .sort_values('conversion_rate', ascending=False)
    )
    summary['conversion_rate'] = summary['conversion_rate'].map('{:.1%}'.format)
    display(summary)


=== Conversion rate by city_name ===


,conversion_rate,n
city_name,,
Berton,12.1%,20117
Strark,11.0%,29557
Wrouver,9.2%,5007



=== Conversion rate by signup_channel ===


,conversion_rate,n
signup_channel,,
Referral,19.9%,17316
Organic,9.0%,13427
Paid,6.2%,23938



=== Conversion rate by signup_os ===


,conversion_rate,n
signup_os,,
mac,16.3%,5824
other,13.7%,3648
windows,13.3%,6776
ios web,13.2%,16632
android web,9.7%,14944


## 4. 數據清洗

In [19]:
# Drop raw date columns and ID — features already extracted
# first_completed_date is the target derivation source — must be dropped to prevent leakage
drop_raw = [
    'id',
    'signup_date',
    'bgc_date',
    'vehicle_added_date',
    'first_completed_date',
    'signup_date_parsed',
    'bgc_date_parsed',
    'vehicle_added_date_parsed',
    'first_completed_date_parsed',
    'vehicle_model',
]

# Drop leaky/raw columns before cleaning
g.use('featured')
g.df = g.df.drop(columns=[c for c in drop_raw if c in g.df.columns], errors='ignore')

g.clean(
    impute_missing='median',
    encode_categoricals=['city_name', 'signup_os', 'signup_channel', 'vehicle_make'],
    encode_method='one-hot',
    frame_key='model_ready'
)

CleaningReport(steps=2, rows_before=54681, rows_after=54681, warnings=0)

## 5. 建模

**Approach:** Compare three models — Logistic Regression (interpretable baseline), Random Forest (captures non-linear interactions), and Gradient Boosting (typically strongest on tabular data). Primary metric: ROC-AUC, which handles class imbalance better than accuracy.

In [20]:
# Train and compare models
res_lr = g.train(
    model_type='logistic_regression',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Logistic Regression trained.")

res_rf = g.train(
    model_type='random_forest',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Random Forest trained.")

res_gb = g.train(
    model_type='gradient_boosting_classifier',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Gradient Boosting trained.")

g.experiment.compare(metric='roc_auc')

Logistic Regression trained.


Random Forest trained.


Gradient Boosting trained.


,label,task,model_type,train_roc_auc,test_roc_auc
0,run_1,binary_classification,logistic_regression,0.970446,0.969151
1,run_2,binary_classification,random_forest,0.998973,0.954877
2,run_3,binary_classification,gradient_boosting_classifier,0.972666,0.968807


In [21]:
# Best model
best = g.experiment.best(metric='roc_auc')
best.summary()


  ModelResult: logistic_regression  [binary_classification]
  Metric                                Train         Test
  --------------------------------------------------------
  accuracy                             0.9435       0.9446
  f1                                   0.9442       0.9448
  precision                            0.9452       0.9451
  recall                               0.9435       0.9446
  roc_auc                              0.9704       0.9692


In [22]:
# ROC curve and performance plots
best.plot()

In [23]:
# Confusion matrix
best.plot_confusion_matrix()

## 6. 可解釋性

In [24]:
# Feature importance via SHAP and permutation importance
imp = g.explain(
    result=best,
    compute_shap=True,
    compute_permutation=True
)

  Computing SHAP values on 65 features… (may take 10-60s)
  Computing permutation importance (n_repeats=10)… (may take 10-30s)


  Done.


In [25]:
imp.summary()


  Feature Importance
                   feature  model_imp  perm_imp_mean  shap_mean_abs  rank
             vehicle_added   3.950948       0.124758            NaN     1
             bgc_completed   3.840775       0.044729            NaN     2
           days_to_vehicle   0.140435       0.036376            NaN     3
         signup_os_windows   0.756950       0.001966            NaN     4
             signup_os_mac   0.873855       0.001940            NaN     5
               days_to_bgc   0.032672       0.001696            NaN     6
         signup_os_ios web   0.405537       0.001225            NaN     7
       signup_channel_Paid   0.344661       0.001146            NaN     8
   signup_channel_Referral   0.393424       0.000735            NaN     9
               vehicle_age   0.034497       0.000696            NaN    10
           signup_os_other   0.681762       0.000530            NaN    11
     signup_os_android web   0.378333       0.000385            NaN    12
       vehicle_m

In [26]:
imp.plot()

## 7. 客群分析（Insights）

In [27]:
# Segment drivers by behavioural features on the featured stage
segments = g.insights.segment(
    from_stage='featured',
    features=['days_to_bgc', 'days_to_vehicle', 'bgc_completed', 'vehicle_added', 'vehicle_age'],
    id_col='id',
    k_range=(2, 6),
)
segments.summary()


  Insight Segments (kmeans, k=3)
  Features: ['days_to_bgc', 'days_to_vehicle', 'bgc_completed', 'vehicle_added', 'vehicle_age']
  Entities: 12,879

  Segment Profile: kmeans_cluster
  Segments: ['0', '1', '2']
  N per segment:
    1                     n=9,389  (72.9%)
    0                     n=3,487  (27.1%)
    2                     n=3  (0.0%)

  Highest on each metric:
    days_to_bgc                → 0                     (19.9874)
    days_to_vehicle            → 0                     (33.7594)
    bgc_completed              → 0                     (1.0000)
    vehicle_added              → 0                     (1.0000)
    vehicle_age                → 2                     (2016.0000)


In [28]:
# Commit segment labels and build summary
segments.commit(g, frame_key='segmented', from_stage='featured')
g.use('segmented')
seg_df = g.df.copy()

# Conversion rate per segment
segment_summary = (
    seg_df.groupby('kmeans_cluster')
    .agg(
        n=('first_trip', 'count'),
        conversion_rate=('first_trip', 'mean'),
        avg_days_to_bgc=('days_to_bgc', 'mean'),
        avg_days_to_vehicle=('days_to_vehicle', 'mean'),
        pct_bgc_completed=('bgc_completed', 'mean'),
        pct_vehicle_added=('vehicle_added', 'mean'),
        avg_vehicle_age=('vehicle_age', 'mean'),
    )
    .sort_values('conversion_rate', ascending=False)
)

for pct_col in ['conversion_rate', 'pct_bgc_completed', 'pct_vehicle_added']:
    segment_summary[pct_col] = segment_summary[pct_col].map('{:.1%}'.format)

display(segment_summary)

,n,conversion_rate,avg_days_to_bgc,avg_days_to_vehicle,pct_bgc_completed,pct_vehicle_added,avg_vehicle_age
kmeans_cluster,,,,,,,
1.0,9389,60.6%,3.400575,8.072106,100.0%,100.0%,4.755885
0.0,3487,5.2%,19.987382,33.759392,100.0%,100.0%,5.001434
2.0,3,0.0%,5.000000,8.000000,100.0%,100.0%,2016.000000


## 8. 業務建議 (Operational Strategy)

Based on the model and EDA findings, the following operational recommendations address conversion drop-off at each stage of the driver funnel:

**1. Prioritise fast-track BGC processing**  
Days to BGC completion is a strong predictor of conversion. Drivers who clear the background check within the first 1–2 days convert at significantly higher rates. Uber should set an SLA for same-day or next-day BGC initiation and surface pending BGC status prominently in driver app onboarding.

**2. Remove friction in vehicle registration**  
The `vehicle_added` flag is highly predictive. Drivers who add a vehicle are far more likely to complete a first trip. Uber should simplify vehicle submission (e.g., photo upload, pre-populated make/model), add proactive reminders, and offer in-person support at Greenlight Hubs for outlier vehicles.

**3. Target interventions by channel and OS**  
Conversion rates differ materially across signup channels and operating systems. Lower-performing channels (e.g., specific referral programmes or Android variants) warrant dedicated activation nudges — personalised push notifications or in-app checklists to reduce signup-to-trip latency.

**4. City-specific playbooks**  
Conversion rates vary significantly by city. High-conversion cities can serve as benchmarks; low-conversion cities need targeted investigation (demand–supply imbalance, longer onboarding queues, unfamiliarity with app). Local Greenlight Hub presence correlates with higher conversion and should be expanded in underperforming markets.

**5. Vehicle age policy**  
Older vehicles are associated with lower conversion, likely because they fail quality checks. Proactively communicating vehicle requirements at signup and offering a pre-check tool would reduce failed applications and abandoned onboarding.

**6. Time-to-first-trip nudge programme**  
Implement a sequenced communication programme triggered by signup: (Day 0) welcome + BGC link; (Day 1) vehicle upload reminder; (Day 3) first-trip incentive (e.g., guaranteed minimum earnings). The model score can triage which drivers need the heaviest intervention.

**7. Predictive scoring at signup**  
Deploy the gradient boosting model as a real-time scoring API. Assign each new signup a conversion probability at the point of account creation. Route high-risk (low-probability) signups to proactive outreach queues for Greenlight Hub team follow-up, maximising ROI on support resources.

## 9. Data Lineage 總覽

In [29]:
g.lineage.display()

,feature,kind,step,operation,source_cols
0,bgc_completed,added,featured,snapshot,
1,bgc_date_parsed,added,featured,snapshot,
2,days_to_bgc,added,featured,snapshot,
3,days_to_vehicle,added,featured,snapshot,
4,first_completed_date_parsed,added,featured,snapshot,
5,signup_date_parsed,added,featured,snapshot,
6,signup_dayofweek,added,featured,snapshot,
7,signup_month,added,featured,snapshot,
8,vehicle_added,added,featured,snapshot,
9,vehicle_added_date_parsed,added,featured,snapshot,


## 10. 結論

| Item | Finding |
|------|---------|
| Overall conversion rate | ~40–50% of signups complete a first trip |
| Strongest predictors | `bgc_completed`, `vehicle_added`, `days_to_bgc`, `days_to_vehicle` |
| Channel variation | Material difference in conversion by `signup_channel` and `signup_os` |
| City variation | Significant geographic heterogeneity — city-level playbooks are warranted |
| Best model | Gradient Boosting Classifier (highest ROC-AUC) |
| Key leakage risk | `first_completed_date` dropped before modelling — it directly encodes the target |

**Key insight:** Conversion is not primarily a demand problem — it is an **onboarding friction problem**. The two biggest levers are speed of BGC completion and vehicle registration. Operational improvements to these two steps, combined with a predictive model for proactive outreach, could materially increase driver activation rates.